In [20]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider


N = 125
m = 3.32e-27      # Masa H2 (kg)
kB = 1.380649e-23
dt = 1e-12        # 1 ps
L_inicial = 10e-6 

# Listas para historia (últimos 1000 pasos)


# Inicialización de estado
pos = np.random.uniform(0, L_inicial, size=(N, 3))
vel = np.random.normal(0, np.sqrt(kB * 300 / m), size=(N, 3))

def maxwell_boltzmann(v, T):
    return 4*np.pi * (m/(2*np.pi*kB*T))**(3/2) * v**2 * np.exp(-m*v**2/(2*kB*T))


fig = plt.figure(figsize=(15, 10))
plt.subplots_adjust(bottom=0.15, hspace=0.4, wspace=0.3)


ax_3d   = fig.add_subplot(2, 3, 1, projection='3d')
ax_temp = fig.add_subplot(2, 3, 2)
ax_pres = fig.add_subplot(2, 3, 3)
ax_hist = fig.add_subplot(2, 3, 6)


ax_s_temp = plt.axes([0.2, 0.05, 0.25, 0.02])
ax_s_size = plt.axes([0.55, 0.05, 0.25, 0.02])
s_temp = Slider(ax_s_temp, 'Temperatura (K)', 50, 1500, valinit=300)
s_size = Slider(ax_s_size, 'Caja (µm)', 2, 20, valinit=10)

hist_T = []
hist_P = []
hist_K = []
hist_pasos = []

contador = 0
while plt.fignum_exists(fig.number):
    L_lim = s_size.val * 1e-6
    T_target = s_temp.val
    
    # Termostato (Re-escalado de velocidades)
    v_sq = np.sum(vel**2)
    K_total = 0.5 * m * v_sq
    T_actual = (2 * K_total) / (3 * N * kB)
    if T_actual > 0:
        vel *= np.sqrt(T_target / T_actual)

    # Integración y Colisiones
    pos += vel * dt
    fuerza = 0
    for i in range(3):
        # Pared Superior
        over = pos[:, i] > L_lim
        fuerza += np.sum(2 * m * np.abs(vel[over, i]))
        pos[over, i] = L_lim
        vel[over, i] *= -1
        # Pared Inferior
        under = pos[:, i] < 0
        fuerza += np.sum(2 * m * np.abs(vel[under, i]))
        pos[under, i] = 0
        vel[under, i] *= -1

    area = 6 * (L_lim**2)
    presion = fuerza /(area * dt)
    
    hist_T.append(T_target)
    hist_P.append(presion)
    hist_K.append(K_total)
    hist_pasos.append(contador)
    
    if len(hist_T) > 1000:
        hist_T.pop(0); hist_P.pop(0); hist_K.pop(0); hist_pasos.pop(0)

    
    if contador % 20 == 0:
        # 1. Simulación 3D
        ax_3d.cla()
        ax_3d.scatter(pos[:,0], pos[:,1], pos[:,2], s=8, c='pink')
        ax_3d.set_xlim(0, L_lim); ax_3d.set_ylim(0, L_lim); ax_3d.set_zlim(0, L_lim)
        ax_3d.set_title("Movimiento de Partículas")


        ax_temp.cla()
        ax_temp.plot(hist_pasos, hist_T, color='red')
        ax_temp.set_title("Temperatura vs Tiempo")
        ax_temp.set_ylabel("K")


        ax_pres.cla()
        ax_pres.plot(hist_pasos, hist_P, color='green', lw=0.5)
        ax_pres.set_title("Presión vs Tiempo")
        ax_pres.set_ylabel("Pa")


        ax_hist.cla()
        v_mags = np.linalg.norm(vel, axis=1)
        ax_hist.hist(v_mags, bins=15, density=True, color='pink',alpha=0.7)
        v_range = np.linspace(0, np.max(v_mags)*1.3, 100)
        ax_hist.plot(v_range, maxwell_boltzmann(v_range, T_target), color= 'magenta', lw=2)
        ax_hist.set_title("Distribución de Velocidades")
        ax_hist.set_xlabel("|v| (m/s)")

        plt.draw()
        plt.pause(0.001)

    contador += 1

In [22]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider


N = 125
m = 3.32e-27      # Masa H2 (kg)
kB = 1.380649e-23
dt = 1e-12        # 1 ps
L_inicial = 10e-6 

# Listas para historia (últimos 1000 pasos)


# Inicialización de estado
pos = np.random.uniform(0, L_inicial, size=(N, 3))
vel = np.random.normal(0, np.sqrt(kB * 300 / m), size=(N, 3))

def maxwell_boltzmann(v, T):
    return 4*np.pi * (m/(2*np.pi*kB*T))**(3/2) * v**2 * np.exp(-m*v**2/(2*kB*T))

# --- 2. Configuración de la Interfaz (Subplots) ---
fig = plt.figure(figsize=(15, 10))
plt.subplots_adjust(bottom=0.15, hspace=0.4, wspace=0.3)

# Definición de los ejes de los gráficos
ax_3d   = fig.add_subplot(2, 3, 1, projection='3d')
ax_temp = fig.add_subplot(2, 3, 2)
ax_pres = fig.add_subplot(2, 3, 3)
ax_hist = fig.add_subplot(2, 3, 6)

# Configuración de Sliders
ax_s_temp = plt.axes([0.2, 0.05, 0.25, 0.02])
ax_s_size = plt.axes([0.55, 0.05, 0.25, 0.02])
s_temp = Slider(ax_s_temp, 'Temperatura (K)', 50, 1500, valinit=300)
s_size = Slider(ax_s_size, 'Caja (µm)', 2, 20, valinit=10)

hist_T = []
hist_P = []
hist_K = []
hist_pasos = []

# --- 3. Bucle de Simulación ---
contador = 0
while plt.fignum_exists(fig.number):
    L_lim = s_size.val * 1e-6
    T_target = s_temp.val
    
    # Termostato (Re-escalado de velocidades)
    v_sq = np.sum(vel**2)
    K_total = 0.5 * m * v_sq
    T_actual = (2 * K_total) / (3 * N * kB)
    if T_actual > 0:
        vel *= np.sqrt(T_target / T_actual)

    # Integración y Colisiones
    pos += vel * dt
    fuerza = 0
    for i in range(3):
        # Pared Superior
        over = pos[:, i] > L_lim
        fuerza += np.sum(2 * m * np.abs(vel[over, i]))
        pos[over, i] = L_lim
        vel[over, i] *= -1
        # Pared Inferior
        under = pos[:, i] < 0
        fuerza += np.sum(2 * m * np.abs(vel[under, i]))
        pos[under, i] = 0
        vel[under, i] *= -1

    area = 6 * (L_lim**2)
    presion = fuerza /(area * dt)
    
    hist_T.append(T_target)
    hist_P.append(presion)
    hist_K.append(K_total)
    hist_pasos.append(contador)
    
    if len(hist_T) > 1000:
        hist_T.pop(0); hist_P.pop(0); hist_K.pop(0); hist_pasos.pop(0)

    
    if contador % 20 == 0:
        # 1. Simulación 3D
        ax_3d.cla()
        ax_3d.scatter(pos[:,0], pos[:,1], pos[:,2], s=8, c='pink')
        ax_3d.set_xlim(0, L_lim); ax_3d.set_ylim(0, L_lim); ax_3d.set_zlim(0, L_lim)
        ax_3d.set_title("Movimiento de Partículas")


        ax_temp.cla()
        ax_temp.plot(hist_pasos, hist_T, color='red')
        ax_temp.set_title("Temperatura vs Tiempo")
        ax_temp.set_ylabel("K")


        ax_pres.cla()
        ax_pres.plot(hist_pasos, hist_P, color='green', lw=0.5)
        ax_pres.set_title("Presión vs Tiempo")
        ax_pres.set_ylabel("Pa")


        ax_hist.cla()
        v_mags = np.linalg.norm(vel, axis=1)
        ax_hist.hist(v_mags, bins=15, density=True, color='pink',alpha=0.7)
        v_range = np.linspace(0, np.max(v_mags)*1.3, 100)
        ax_hist.plot(v_range, maxwell_boltzmann(v_range, T_target), color= 'magenta', lw=2)
        ax_hist.set_title("Distribución de Velocidades")
        ax_hist.set_xlabel("|v| (m/s)")

        plt.draw()
        plt.pause(0.001)

    contador += 1